# 01 · Exploratory data analysis

In [2]:
from datetime import date

import plotly.io as pio
import polars as pl

from energy_forecast import plots
from energy_forecast.data import ROOT, load

pio.templates.default = "plotly_white"
df = load()
df.tail(3)

time,price,load,lignite,nuclear,hard_coal,gas,other_conventional,pumped_storage,wind_onshore,wind_offshore,solar,biomass,hydro,other_renewable,load_fc,wind_onshore_fc,wind_offshore_fc,solar_fc,Wind,Solar,Biomass,Hydro & other RE,Gas,Hard coal,Lignite,Other conventional,residual_load,residual_load_fc,local
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,datetime[μs]
2026-11-08 20:00:00 UTC,272.71,null,null,0.0,null,null,null,null,null,null,null,null,null,null,52821.49,2622.96,1761.33,0.0,null,null,null,null,null,null,null,null,null,48437.2,2026-11-08 21:00:00
2026-11-08 21:00:00 UTC,253.26,null,null,0.0,null,null,null,null,null,null,null,null,null,null,50755.56,2378.03,1665.99,0.0,null,null,null,null,null,null,null,null,null,46711.54,2026-11-08 22:00:00
2026-11-08 22:00:00 UTC,219.93,null,null,0.0,null,null,null,null,null,null,null,null,null,null,48092.8,2108.43,1515.86,0.0,null,null,null,null,null,null,null,null,null,44468.51,2026-11-08 23:00:00


## Price by hour and month

In [3]:
hero = plots.price_heatmap(df, 2025)
hero

In [4]:
plots.price_heatmap_by_year(df).update_layout(height=700)

## Generation mix: summer vs winter week

In [5]:
plots.mix_area(df, date(2025, 6, 16), date(2025, 6, 22))

In [6]:
plots.mix_area(df, date(2025, 1, 13), date(2025, 1, 19))

## Merit order: residual load vs price

In [7]:
plots.merit_order(df).update_layout(height=700)

## Renewables and negative prices

In [8]:
plots.renewable_share(df)

In [9]:
plots.negative_hours(df)

In [10]:
df.group_by(year=pl.col("local").dt.year()).agg(
    mean_price=pl.col("price").mean(),
    std_price=pl.col("price").std(),
    min_price=pl.col("price").min(),
    max_price=pl.col("price").max(),
    negative_hours=(pl.col("price") < 0).sum(),
).sort("year")

year,mean_price,std_price,min_price,max_price,negative_hours
i32,f64,f64,f64,f64,u32
2019,37.667668,15.518065,-90.01,121.46,211
2020,30.470715,17.500846,-83.94,200.04,298
2021,96.849918,73.680947,-69.0,620.0,139
2022,235.446143,142.809409,-19.04,871.0,69
2023,95.175452,47.584215,-500.0,524.27,301
2024,78.512033,52.726409,-135.45,936.28,457
2025,89.321912,52.101572,-250.32,583.4,573
2026,117.657162,72.970769,-499.0,697.31,462


## Export README figures

In [11]:
out = ROOT / "docs/img"
out.mkdir(parents=True, exist_ok=True)
hero.write_image(out / "price_heatmap.png", width=900, height=560, scale=2)